<a href="https://colab.research.google.com/github/Epidemic2001/PetProyects_Studies/blob/main/02_Transfer_Learning_CIFAR10_MobileNetV2_EXPLICADO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 2 — Transfer Learning con MobileNetV2 y CIFAR-10

## Objetivo general
En este laboratorio aprenderemos a reutilizar el conocimiento de una red neuronal previamente entrenada para resolver una nueva tarea de clasificación.

Utilizaremos **MobileNetV2**, una red convolucional entrenada previamente con **ImageNet**, y la adaptaremos al dataset **CIFAR-10**.

La idea central será:

`conocimiento previo → modelo preentrenado → nueva tarea`

## Conceptos que aparecerán
- **Transfer Learning:** reutilización de conocimiento aprendido en una tarea para resolver otra.
- **Modelo preentrenado:** modelo cuyos pesos ya fueron ajustados usando otro dataset.
- **Feature Extraction:** uso del modelo preentrenado como extractor de características sin modificar sus pesos.
- **Fine-Tuning:** ajuste controlado de algunas capas del modelo preentrenado.
- **Learning rate:** tamaño de los cambios aplicados a los pesos durante el entrenamiento.
- **Data augmentation:** generación de variaciones artificiales de las imágenes para mejorar generalización.

## Qué deberá comprender el alumno
1. Por qué no siempre es necesario entrenar desde cero.
2. Qué significa congelar capas.
3. Diferencia entre feature extraction y fine-tuning.
4. Por qué el fine-tuning suele utilizar un learning rate pequeño.
5. Cómo evaluar si la transferencia mejora el modelo.

# 1. Importar librerías

Cargaremos las herramientas necesarias para trabajar con redes neuronales, imágenes, gráficos y métricas.

- **NumPy:** operaciones numéricas.
- **Matplotlib:** visualización.
- **TensorFlow/Keras:** construcción y entrenamiento del modelo.
- **Scikit-learn:** reporte de métricas de clasificación.

También fijaremos semillas aleatorias para obtener ejecuciones más reproducibles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report

print("TensorFlow:", tf.__version__)

np.random.seed(42)
tf.random.set_seed(42)

# 2. Cargar el dataset CIFAR-10

**CIFAR-10** es un dataset clásico de visión por computador con 60.000 imágenes RGB de tamaño 32 × 32 píxeles.

Está dividido en 10 clases:
avión, automóvil, pájaro, gato, ciervo, perro, rana, caballo, barco y camión.

### ¿Qué significa RGB?
Cada imagen posee tres canales de color: **Red, Green y Blue**.

### División típica
- 50.000 imágenes para entrenamiento.
- 10.000 imágenes para prueba.

El conjunto de prueba se reserva para evaluar el desempeño final del modelo.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "avión", "automóvil", "pájaro", "gato", "ciervo",
    "perro", "rana", "caballo", "barco", "camión"
]

print("Entrenamiento:", x_train.shape)
print("Prueba:", x_test.shape)

## 3. Visualizar el dataset

Antes de entrenar cualquier modelo es buena práctica observar los datos.

CIFAR-10 tiene imágenes pequeñas y de baja resolución, por lo que algunos objetos pueden resultar difíciles de identificar incluso para una persona.

### Qué observar
- variedad de clases,
- calidad de las imágenes,
- diferencias entre categorías visualmente parecidas.

In [ ]:
plt.figure(figsize=(10, 6))

for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

# 4. ¿Qué es Transfer Learning?

**Transfer Learning** consiste en tomar un modelo que ya aprendió patrones útiles en un problema grande y reutilizar ese conocimiento en una nueva tarea.

En visión por computador, las primeras capas de una CNN suelen aprender características generales como:
- bordes,
- esquinas,
- texturas,
- formas simples.

Las capas más profundas aprenden combinaciones más específicas de esas características.

### Ventaja
En lugar de comenzar con pesos aleatorios, comenzamos con un modelo que ya posee conocimiento visual.

### Modelo utilizado: MobileNetV2
**MobileNetV2** es una CNN diseñada para ser eficiente en memoria y cálculo. Fue pensada especialmente para dispositivos con recursos limitados y utiliza bloques convolucionales eficientes llamados **inverted residuals** y **depthwise separable convolutions**.

No es necesario comprender toda su arquitectura interna para usarla en este laboratorio. Lo importante es que ya fue entrenada sobre ImageNet y contiene representaciones visuales útiles.

# 5. Preparar las imágenes

MobileNetV2 fue entrenada con imágenes de mayor tamaño que CIFAR-10, por lo que necesitamos adaptar nuestros datos.

### Redimensionamiento
Cambiaremos cada imagen desde `32 × 32` a `96 × 96`.

### Data Augmentation
Aplicaremos transformaciones aleatorias como:
- volteo horizontal,
- pequeñas rotaciones,
- zoom.

**Data augmentation** crea variaciones realistas de las imágenes de entrenamiento. Esto ayuda al modelo a no memorizar ejemplos exactos y mejora su capacidad de generalización.

In [ ]:
IMG_SIZE = 96
NUM_CLASSES = 10

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="data_augmentation")

# 6. Cargar MobileNetV2 preentrenada

Utilizaremos la versión disponible en Keras.

Dos parámetros son especialmente importantes:

### `weights="imagenet"`
Indica que cargaremos los pesos obtenidos al entrenar el modelo con ImageNet.

### `include_top=False`
Elimina la capa clasificadora original de ImageNet.

Esto es necesario porque nuestra nueva tarea tiene 10 clases distintas.

Conservaremos el extractor de características y construiremos un clasificador nuevo sobre él.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

print("Número de capas:", len(base_model.layers))

# 7. Feature Extraction: congelar el modelo base

La primera estrategia será usar MobileNetV2 solamente como **extractor de características**.

### ¿Qué significa congelar una capa?
Significa que sus pesos no se modificarán durante el entrenamiento.

Al ejecutar:

`base_model.trainable = False`

preservamos todo el conocimiento aprendido previamente en ImageNet.

Solo se entrenarán las capas nuevas que agregaremos después.

Esta estrategia es útil cuando:
- tenemos pocos datos,
- queremos entrenar rápido,
- la nueva tarea es razonablemente parecida a la tarea original.

In [ ]:
base_model.trainable = False

print("¿MobileNetV2 entrenable?:", base_model.trainable)

# 8. Construir el nuevo clasificador

Sobre MobileNetV2 agregaremos una pequeña cabeza de clasificación.

La arquitectura será:

`imagen → augmentation → resize → MobileNetV2 → pooling → dropout → softmax`

### Componentes
- **Resizing:** adapta el tamaño de entrada.
- **preprocess_input:** transforma los valores al formato esperado por MobileNetV2.
- **GlobalAveragePooling2D:** resume cada mapa de características en un valor promedio.
- **Dropout:** desactiva aleatoriamente algunas neuronas durante el entrenamiento para reducir overfitting.
- **Dense + Softmax:** produce las probabilidades de las 10 clases.

### Softmax
Convierte los valores de salida en probabilidades cuya suma es aproximadamente 1.

In [ ]:
inputs = keras.Input(shape=(32, 32, 3))

x = data_augmentation(inputs)
x = layers.Resizing(IMG_SIZE, IMG_SIZE)(x)
x = keras.applications.mobilenet_v2.preprocess_input(x)

# training=False mantiene Batch Normalization en modo inferencia
x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="clasificador_cifar10"
)(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# 9. Entrenamiento — Fase 1: Feature Extraction

Entrenaremos solamente las capas nuevas del clasificador.

MobileNetV2 permanece congelada.

### ¿Por qué usamos solo una muestra?
Para que el laboratorio sea ejecutable con rapidez incluso en Colab sin GPU.

Si se dispone de GPU puede aumentarse `N_TRAIN` hasta usar todo el conjunto de entrenamiento.

### Qué aprende esta fase
El clasificador aprende a convertir las características generales producidas por MobileNetV2 en las 10 clases de CIFAR-10.

In [ ]:
N_TRAIN = 15000

history_feature = model.fit(
    x_train[:N_TRAIN],
    y_train[:N_TRAIN],
    epochs=4,
    batch_size=64,
    validation_split=0.15,
    verbose=1
)

# 10. Analizar las curvas de aprendizaje

Graficaremos la **accuracy** del entrenamiento y de validación.

### Accuracy
Es la proporción de predicciones correctas.

### Validation set
Es un subconjunto separado durante el entrenamiento que permite evaluar si el modelo generaliza a datos no vistos.

Si la accuracy de entrenamiento aumenta mientras la de validación se estanca o disminuye, puede existir overfitting.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_feature.history["accuracy"], label="Train")
plt.plot(history_feature.history["val_accuracy"], label="Validation")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Feature Extraction con MobileNetV2")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 11. Evaluar la primera fase

Ahora mediremos el desempeño con el conjunto de prueba completo.

Este resultado servirá como referencia para saber si el fine-tuning posterior aporta una mejora.

In [ ]:
test_loss_before, test_acc_before = model.evaluate(
    x_test,
    y_test,
    batch_size=64,
    verbose=0
)

print(f"Accuracy antes del fine-tuning: {test_acc_before:.4f}")

# 12. Fine-Tuning

**Fine-tuning** significa permitir que algunas capas del modelo preentrenado vuelvan a aprender, pero de forma controlada.

La idea es adaptar las características más profundas a CIFAR-10 sin destruir el conocimiento general aprendido previamente.

### Estrategia
1. Activamos nuevamente `base_model.trainable = True`.
2. Mantenemos congelada la mayor parte de las capas.
3. Liberamos solo las últimas capas.

Las capas finales suelen contener características más específicas, por lo que son buenas candidatas para adaptarse a la nueva tarea.

In [ ]:
base_model.trainable = True

# Congelamos la mayor parte de MobileNetV2
fine_tune_at = len(base_model.layers) - 25

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print("Total de capas:", len(base_model.layers))
print("Capas aproximadamente liberadas:", 25)

## 13. Recompilar con un learning rate pequeño

Cuando cambiamos qué capas son entrenables debemos volver a compilar el modelo.

### Learning rate
El **learning rate** controla cuánto cambian los pesos en cada actualización.

Para fine-tuning usamos un valor mucho menor porque:
- los pesos ya contienen conocimiento útil,
- queremos hacer ajustes suaves,
- cambios demasiado grandes podrían degradar el modelo.

Este fenómeno se relaciona con el llamado **catastrophic forgetting**, donde una red pierde conocimiento previamente aprendido al ser modificada demasiado agresivamente.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 14. Entrenar la fase de fine-tuning

En esta fase se actualizan:
- las capas nuevas del clasificador,
- algunas de las últimas capas de MobileNetV2.

El objetivo es adaptar mejor las características de alto nivel a las imágenes de CIFAR-10.

In [ ]:
history_fine = model.fit(
    x_train[:N_TRAIN],
    y_train[:N_TRAIN],
    epochs=3,
    batch_size=64,
    validation_split=0.15,
    verbose=1
)

# 15. Evaluar después del fine-tuning

Compararemos la accuracy antes y después del fine-tuning.

### Interpretación
Si la accuracy aumenta, el ajuste fino probablemente ayudó a adaptar el modelo.

Si disminuye, podría deberse a:
- learning rate demasiado alto,
- exceso de fine-tuning,
- pocos datos,
- sobreajuste.

In [ ]:
test_loss_after, test_acc_after = model.evaluate(
    x_test,
    y_test,
    batch_size=64,
    verbose=0
)

print(f"Accuracy antes del fine-tuning: {test_acc_before:.4f}")
print(f"Accuracy después del fine-tuning: {test_acc_after:.4f}")

# 16. Comparación gráfica

Mostraremos lado a lado el desempeño obtenido con:
- Feature Extraction.
- Fine-Tuning.

La gráfica permite interpretar rápidamente si la segunda estrategia produjo una mejora real.

In [ ]:
labels = ["Feature Extraction", "Fine-Tuning"]
values = [test_acc_before, test_acc_after]

plt.figure(figsize=(7, 4))
plt.bar(labels, values)
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Transfer Learning: comparación de estrategias")

for i, value in enumerate(values):
    plt.text(i, value + 0.02, f"{value:.3f}", ha="center")

plt.show()

# 17. Clasificar imágenes individuales

Además de mirar métricas globales, es útil inspeccionar predicciones concretas.

Mostraremos:
- clase real,
- clase predicha.

Esto ayuda a detectar qué tipos de objetos confunde el modelo.

In [ ]:
pred_probs = model.predict(x_test[:15], verbose=0)
pred_classes = np.argmax(pred_probs, axis=1)

plt.figure(figsize=(12, 8))

for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[i])

    real = class_names[y_test[i]]
    predicted = class_names[pred_classes[i]]

    plt.title(f"Real: {real}\nPred: {predicted}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# 18. Reporte de clasificación

La accuracy por sí sola no siempre cuenta toda la historia.

Calcularemos también:
- **Precision:** qué tan confiables son las predicciones positivas de cada clase.
- **Recall:** qué proporción de los ejemplos reales de una clase fueron detectados.
- **F1-score:** balance entre precision y recall.

Estas métricas permiten analizar el comportamiento por clase.

In [ ]:
pred_probs_all = model.predict(x_test, batch_size=64, verbose=0)
y_pred = np.argmax(pred_probs_all, axis=1)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names
    )
)

# 19. Conclusiones del laboratorio

Hemos probado dos formas de Transfer Learning.

## Estrategia A — Feature Extraction
`MobileNetV2 congelada → nuevo clasificador`

Se aprovecha el conocimiento previo sin modificarlo.

## Estrategia B — Fine-Tuning
`MobileNetV2 parcialmente entrenable → adaptación a CIFAR-10`

Se ajustan algunas capas para especializar el modelo.

## Conceptos clave
| Concepto | Descripción breve |
|---|---|
| Transfer Learning | reutilización de conocimiento previamente aprendido |
| Pretrained model | modelo con pesos ya entrenados |
| Feature Extraction | uso del modelo como extractor fijo de características |
| Fine-Tuning | ajuste controlado de algunas capas preentrenadas |
| Frozen layer | capa cuyos pesos no se actualizan |
| Learning rate | magnitud de cada actualización de pesos |
| Data augmentation | creación de variaciones artificiales de los datos |
| Softmax | función que produce probabilidades entre varias clases |
| Dropout | regularización que desactiva temporalmente neuronas |
| Overfitting | aprendizaje excesivo del entrenamiento con peor generalización |

## Preguntas para discusión
1. ¿Qué significa `weights="imagenet"`?
2. ¿Por qué usamos `include_top=False`?
3. ¿Qué diferencia existe entre feature extraction y fine-tuning?
4. ¿Por qué el fine-tuning utiliza un learning rate pequeño?
5. ¿Cuándo podría ser mejor entrenar desde cero?
6. ¿Qué ocurriría si la nueva tarea fuera completamente distinta de ImageNet?